# Attention Mechanism: The Transformer Core

Reach for this when you need: 
- Reference for Dot-Product and Multi-Head Attention (MHA).
- To understand Queries, Keys, and Values (QKV).
- Reference for scaled dot-product math.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Scaled Dot-Product Attention

Equation: $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

| Component | Description | Usage |
| :--- | :--- | :--- |
| `Query (Q)` | "What I'm looking for" | Searching through context |
| `Key (K)` | "What I offer" | Matching against search query |
| `Value (V)` | "What I am" | The actual information being extracted |
| `Scale` | $1/\sqrt{d_k}$ | Preventing softmax saturation (vanishing gradients) |

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    # Matmul + scale
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        # mask=0: keep; mask=1: block (set to -inf)
        scores = scores.masked_fill(mask == 1, -1e9)
        
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, v), weights

## 2. Multi-Head Attention (nn.MultiheadAttention)
Instead of one complex attention, perform multiple simple attentions in parallel across lower dimensions.

✅ **Use when**: Capturing multiple relationships (e.g. one head for grammar, one for coreference).
❌ **Don't use when**: Extreme low-latency is needed (use a single head or FlashAttention).

In [ ]:
# nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
mha = nn.MultiheadAttention(embed_dim=768, num_heads=12, batch_first=True)

query = torch.randn(1, 10, 768) # Batch=1, Seq=10
key = torch.randn(1, 10, 768)
value = torch.randn(1, 10, 768)

attn_output, attn_weights = mha(query, key, value)

### Common Pitfalls
- **Normalization Scale**: Forgetting to divide by $\sqrt{d_k}$. Without this, the softmax output becomes biased toward a single token (delta distribution).
- **Masking Logic**: In causal masking (decoder), each token can only see previous tokens. Ensure the mask is correctly applied to the upper triangle of the scores matrix.
- **Batching**: MultiheadAttention expects 3D inputs. Ensure you have the `(Batch, Seq, Features)` shape if `batch_first=True`.

### Key Takeaways
- Scaled dot-product is the mathematical engine of the Transformer.
- `Multi-Head` allows the model to project embeddings into different subspaces simultaneously.
- Attention is $O(N^2)$ in time and space complexity with respect to sequence length $N$.